# 🧬 Pfizer Document Intelligence — RAG Chatbot
**Externship Project | Document Scanning & AI Chatbot**

This notebook builds a complete Retrieval-Augmented Generation (RAG) pipeline that:
- Processes both digital and scanned PDFs
- Tags chunks with metadata (document type, page, source)
- Embeds and indexes content with FAISS
- Retrieves relevant context and generates grounded answers
- Displays a Gradio chat UI with sources and confidence scores

---
**Pipeline Overview:**
```
Upload PDF → OCR (if scanned) → Chunk + Tag → Embed → FAISS Index
                                                              ↓
User Query → Embed Query → Retrieve Top-K Chunks → Build Prompt → LLM → Answer + Sources
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Install Dependencies
We install all required packages:
- `PyMuPDF` (fitz) — extract text from digital PDFs
- `pytesseract` + `pdf2image` — OCR for scanned PDFs
- `sentence-transformers` — open-source embedding model
- `faiss-cpu` — vector similarity search
- `transformers` — open-source LLM (Flan-T5)
- `gradio` — UI framework

In [2]:
# Install all required packages
!pip install pymupdf pytesseract pdf2image pillow -q
!pip install sentence-transformers faiss-cpu -q
!pip install transformers accelerate -q
!pip install gradio -q

# Install system-level Tesseract OCR engine (needed for scanned PDFs)
!apt-get install -y tesseract-ocr poppler-utils -q

print("✅ All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (425 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for m

## Step 2: Import Libraries and Configure Settings

In [3]:
import os
import re
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# PDF processing
import fitz  # PyMuPDF for digital PDFs
import pytesseract  # OCR engine
from pdf2image import convert_from_path  # Convert PDF pages to images for OCR
from PIL import Image

# ML / embeddings / indexing
from sentence_transformers import SentenceTransformer
import faiss

# Open-source LLM (Flan-T5 — instruction-tuned, runs on CPU/GPU)
from transformers import pipeline as hf_pipeline

# UI
import gradio as gr

# ── Configuration ──────────────────────────────────────────────────────────────
CHUNK_SIZE = 400          # Characters per text chunk
CHUNK_OVERLAP = 80        # Character overlap between chunks (for context continuity)
TOP_K = 4                 # Number of chunks to retrieve per query
OCR_CONFIDENCE_THRESHOLD = 60  # Minimum Tesseract confidence to accept a page as digital
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"  # Lightweight but strong sentence-transformers model
LLM_MODEL_NAME = "google/flan-t5-base"  # Open-source instruction-tuned LLM

print("✅ Libraries imported and configuration set.")
print(f"   Embedding model : {EMBED_MODEL_NAME}")
print(f"   LLM             : {LLM_MODEL_NAME}")
print(f"   Chunk size      : {CHUNK_SIZE} chars (overlap {CHUNK_OVERLAP})")
print(f"   Retrieval top-K : {TOP_K}")

✅ Libraries imported and configuration set.
   Embedding model : all-MiniLM-L6-v2
   LLM             : google/flan-t5-base
   Chunk size      : 400 chars (overlap 80)
   Retrieval top-K : 4


## Step 3: Document Processing — Digital PDFs + OCR for Scanned Files

**How it works:**
1. First we try to extract text directly from the PDF using PyMuPDF (fast, accurate for digital PDFs).
2. If a page yields very little text (< 30 chars), we assume it's a scanned image and run Tesseract OCR on that page.
3. Each page's text is labeled with the extraction method used.

In [4]:
def extract_text_from_pdf(pdf_path: str) -> List[Dict]:
    """
    Extract text from a PDF file, applying OCR for scanned pages.

    Returns a list of dicts, one per page:
        {
            'page': int,
            'text': str,
            'method': 'digital' | 'ocr',
            'char_count': int
        }
    """
    pages_data = []

    # Open the PDF with PyMuPDF
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    print(f"📄 Processing '{Path(pdf_path).name}' — {total_pages} page(s)")

    for page_num in range(total_pages):
        page = doc[page_num]

        # --- Attempt 1: Extract digital text directly ---
        digital_text = page.get_text("text").strip()

        if len(digital_text) >= 30:
            # Page has enough digital text — use it directly
            method = "digital"
            final_text = digital_text
        else:
            # --- Attempt 2: OCR the page as an image ---
            print(f"   Page {page_num+1}: sparse text ({len(digital_text)} chars) → applying OCR...")
            # Render page to a high-resolution image (300 DPI for good OCR accuracy)
            mat = fitz.Matrix(300 / 72, 300 / 72)  # scale factor for 300 DPI
            pix = page.get_pixmap(matrix=mat)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            # Run Tesseract OCR; get both text and confidence data
            ocr_data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
            # Filter words above confidence threshold
            words = [
                ocr_data['text'][i]
                for i in range(len(ocr_data['text']))
                if int(ocr_data['conf'][i]) > OCR_CONFIDENCE_THRESHOLD
                and ocr_data['text'][i].strip()
            ]
            final_text = " ".join(words)
            method = "ocr"

        # Clean up whitespace
        final_text = re.sub(r'\s+', ' ', final_text).strip()

        pages_data.append({
            'page': page_num + 1,  # 1-indexed for human readability
            'text': final_text,
            'method': method,
            'char_count': len(final_text)
        })

    doc.close()
    digital_pages = sum(1 for p in pages_data if p['method'] == 'digital')
    ocr_pages = sum(1 for p in pages_data if p['method'] == 'ocr')
    print(f"   ✅ Extracted: {digital_pages} digital page(s), {ocr_pages} OCR page(s)")
    return pages_data


print("✅ extract_text_from_pdf() defined.")

✅ extract_text_from_pdf() defined.


## Step 4: Text Chunking + Metadata Tagging

We split each page's text into overlapping chunks and attach metadata to each one.

**Metadata fields per chunk:**
- `source` — filename
- `doc_type` — inferred category (clinical, regulatory, label, research, general)
- `page` — page number in the source PDF
- `chunk_id` — sequential ID for retrieval
- `extraction_method` — 'digital' or 'ocr'

In [5]:
# Keyword mapping to infer document type from filename or content
DOC_TYPE_KEYWORDS = {
    "clinical": ["trial", "study", "patient", "efficacy", "adverse", "dosage"],
    "regulatory": ["fda", "approval", "submission", "anda", "nda", "ema", "510k"],
    "label":     ["label", "prescribing", "indication", "contraindication", "warning"],
    "research":  ["research", "mechanism", "pharmacokinetics", "in vitro", "in vivo"],
}


def infer_doc_type(filename: str, text_sample: str) -> str:
    """Guess document type from filename and first 500 chars of content."""
    combined = (filename + " " + text_sample[:500]).lower()
    for doc_type, keywords in DOC_TYPE_KEYWORDS.items():
        if any(kw in combined for kw in keywords):
            return doc_type
    return "general"


def chunk_pages(pages_data: List[Dict], source_name: str) -> List[Dict]:
    """
    Split page text into overlapping chunks and attach metadata.

    Strategy:
    - Iterate over each page's text in windows of CHUNK_SIZE chars
    - Overlap chunks by CHUNK_OVERLAP chars to avoid splitting key sentences
    - Each chunk carries: text, source, page, chunk_id, doc_type, extraction_method
    """
    chunks = []
    chunk_id = 0

    # Infer document type from filename + first page text
    first_text = pages_data[0]['text'] if pages_data else ""
    doc_type = infer_doc_type(source_name, first_text)

    for page_data in pages_data:
        text = page_data['text']
        if not text:
            continue  # Skip empty pages

        # Slide a window across the page text
        start = 0
        while start < len(text):
            end = min(start + CHUNK_SIZE, len(text))
            chunk_text = text[start:end].strip()

            if len(chunk_text) > 20:  # Skip near-empty chunks
                chunks.append({
                    "chunk_id":          chunk_id,
                    "text":              chunk_text,
                    "source":            source_name,
                    "page":              page_data['page'],
                    "doc_type":          doc_type,
                    "extraction_method": page_data['method'],
                    "char_count":        len(chunk_text),
                })
                chunk_id += 1

            # Move window forward by (CHUNK_SIZE - CHUNK_OVERLAP) for overlap
            start += CHUNK_SIZE - CHUNK_OVERLAP

    print(f"   📦 Created {len(chunks)} chunks from '{source_name}' (type: {doc_type})")
    return chunks


print("✅ chunk_pages() and infer_doc_type() defined.")

✅ chunk_pages() and infer_doc_type() defined.


## Step 5: Embedding Model + FAISS Index

We use **sentence-transformers** (`all-MiniLM-L6-v2`) to convert text chunks into dense vector embeddings, then store them in a **FAISS** flat L2 index for fast nearest-neighbor retrieval.

FAISS is efficient even for thousands of documents and runs entirely locally — no API keys needed.

In [6]:
# Load the embedding model (downloads ~80MB on first run, then cached)
print(f"⏳ Loading embedding model '{EMBED_MODEL_NAME}'...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"✅ Embedding model loaded. Vector dimension: {EMBED_DIM}")

# Global state: all chunks and their FAISS index
# These are populated each time a new document is uploaded
all_chunks: List[Dict] = []    # list of chunk dicts with metadata
faiss_index = None             # FAISS index (built/rebuilt when docs are added)


def build_faiss_index(chunks: List[Dict]) -> faiss.IndexFlatL2:
    """
    Embed all chunks and build a FAISS L2 index.
    - Uses batch encoding for efficiency
    - Returns the index so it can be stored globally
    """
    texts = [c['text'] for c in chunks]
    print(f"⏳ Embedding {len(texts)} chunks...")

    # Encode in batches of 64 (memory-friendly)
    embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)

    # FAISS expects float32
    embeddings = np.array(embeddings, dtype=np.float32)

    # Build a flat (exact) L2 index — good for up to ~50k chunks
    index = faiss.IndexFlatL2(EMBED_DIM)
    index.add(embeddings)
    print(f"✅ FAISS index built — {index.ntotal} vectors stored.")
    return index


print("✅ build_faiss_index() defined.")

⏳ Loading embedding model 'all-MiniLM-L6-v2'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded. Vector dimension: 384
✅ build_faiss_index() defined.


/tmp/ipykernel_1776/775229927.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embed_model.get_sentence_embedding_dimension()


## Step 6: Load the Open-Source LLM

We use **Flan-T5-base** from Google via Hugging Face Transformers:
- Instruction-tuned, so it follows natural language prompts well
- Runs on CPU without needing a GPU
- Free and open-source — no API key required
- Generates concise, factual answers suitable for document Q&A

In [7]:
# Load the LLM using AutoModelForSeq2SeqLM + AutoTokenizer directly.
# We avoid hf_pipeline() here because newer versions of transformers removed
# the 'text2text-generation' task alias. Calling the model directly is more
# explicit, version-stable, and gives us full control over decoding.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print(f"⏳ Loading LLM '{LLM_MODEL_NAME}' (this may take ~1–2 minutes on first run)...")

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model     = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME)
llm_model.eval()  # Put model in inference mode (disables dropout)

# Detect device: use GPU if available (Colab T4), otherwise CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(DEVICE)
print(f"   Running on: {DEVICE.upper()}")

def llm_generate(prompt: str, max_new_tokens: int = 300) -> str:
    """
    Tokenize a prompt and generate a response with Flan-T5.
    Returns the decoded output string.
    """
    # Tokenize and move to device; truncate to 512 tokens (T5-base encoder limit)
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():  # Disable gradient computation for speed
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # Greedy decoding — deterministic output
            num_beams=2,       # Mild beam search for better answer coherence
        )

    # Decode the generated token IDs back to text
    return llm_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print(f"✅ LLM loaded: {LLM_MODEL_NAME}")
print("   Quick smoke test:", llm_generate("What is a clinical trial? Answer in one sentence."))

⏳ Loading LLM 'google/flan-t5-base' (this may take ~1–2 minutes on first run)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

   Running on: CUDA
✅ LLM loaded: google/flan-t5-base
   Quick smoke test: A clinical trial is a medical trial that evaluates the effectiveness of a drug.


## Step 7: Retrieval Function

Given a user query, we:
1. Embed the query with the same model used for chunks
2. Search FAISS for the top-K nearest chunk vectors
3. Return the matching chunks with **confidence scores** derived from L2 distance

Confidence score: `max(0, 1 - distance / scaling_constant)` — higher is better.

In [31]:
def retrieve_chunks(
    query: str,
    k: int = TOP_K,
    doc_type_filter: Optional[str] = None
) -> List[Dict]:
    """
    Retrieve the top-k most relevant chunks for a query.

    Args:
        query            : The user's question
        k                : Number of chunks to retrieve
        doc_type_filter  : If provided, only return chunks of this doc_type

    Returns:
        List of chunk dicts, each augmented with a 'confidence' score (0–1)
    """
    global faiss_index, all_chunks

    if faiss_index is None or len(all_chunks) == 0:
        return []  # No documents indexed yet

    # Embed the query (same model as chunks)
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)

    # Retrieve more candidates if we're going to filter
    search_k = k * 3 if doc_type_filter else k
    distances, indices = faiss_index.search(query_vec, min(search_k, len(all_chunks)))

    print("Raw FAISS indices:", indices[0])

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < 0:  # FAISS returns -1 for empty slots
            continue

        chunk = all_chunks[idx].copy()

        # Apply optional doc_type filter
        if doc_type_filter and chunk.get('doc_type') != doc_type_filter:
            continue

        # Convert L2 distance to a 0–1 confidence score
        # Lower distance = higher confidence; we scale by 20 (tuned empirically)
        confidence = float(max(0.0, 1.0 - dist / 20.0))
        chunk['confidence'] = round(confidence, 3)
        chunk['l2_distance'] = round(float(dist), 4)

        results.append(chunk)
        if len(results) >= k:
            break

    # Deduplicate overlapping chunks before returning
    seen = set()
    deduped = []
    for chunk in results:
        key = chunk['text'][:200]
        if key not in seen:
            seen.add(key)
            deduped.append(chunk)
    return deduped

print("✅ retrieve_chunks() defined.")

✅ retrieve_chunks() defined.


## Step 8: Prompt Builder

We construct a clear, structured prompt that:
- Provides the retrieved context chunks
- Instructs the model to answer based only on the provided context
- Asks the model to mention sources (document name + page number) in its answer
- Tells the model to say "I don't know" if the answer isn't in the context

This grounding prevents hallucinations and ensures traceable answers.

In [9]:
def build_prompt(query: str, retrieved_chunks: List[Dict]) -> str:
    """
    Construct a grounded prompt for the LLM.

    Structure:
      [System instruction]
      Context 1: [source, page] — [text]
      Context 2: ...
      Question: ...
      Answer:
    """
    if not retrieved_chunks:
        return f"Answer the following question: {query}\nAnswer: I don't have enough information to answer this question."

    # Build context block from retrieved chunks
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        source_label = f"{chunk['source']} (page {chunk['page']})"
        context_parts.append(f"Context {i} [{source_label}]: {chunk['text']}")

    context_block = "\n\n".join(context_parts)

    # Flan-T5 responds well to explicit instruction prompts
    prompt = (
        "You are a Pfizer document assistant. Answer the question using ONLY the provided context.\n"
        "Cite the source name and page number when referencing information.\n"
        "If the answer is not in the context, say: 'This information is not found in the uploaded documents.'\n\n"
        f"{context_block}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt


print("✅ build_prompt() defined.")

✅ build_prompt() defined.


## Step 9: Complete RAG Pipeline

This function ties everything together:
```
Query → Retrieve → Build Prompt → LLM → Answer + Source Citations + Confidence Scores
```

In [10]:
def run_rag_pipeline(query: str, doc_type_filter: Optional[str] = None) -> Dict:
    """
    Full RAG pipeline: retrieve relevant chunks → build prompt → generate answer.

    Returns a dict with:
        'answer'       : Generated text from the LLM
        'sources'      : List of source references (filename + page)
        'chunks'       : The retrieved chunk dicts (with confidence scores)
        'chunk_count'  : Number of chunks used
        'avg_confidence': Average confidence score across retrieved chunks
    """
    # --- Stage 1: Retrieve relevant chunks ---
    retrieved = retrieve_chunks(query, k=TOP_K, doc_type_filter=doc_type_filter)

    if not retrieved:
        return {
            'answer': "No documents have been uploaded yet, or no relevant content was found.",
            'sources': [],
            'chunks': [],
            'chunk_count': 0,
            'avg_confidence': 0.0
        }

    # --- Stage 2: Build the grounded prompt ---
    prompt = build_prompt(query, retrieved)

    # --- Stage 3: Generate answer with the LLM ---
    # llm_generate() handles tokenization, device placement, and decoding internally
    answer = llm_generate(prompt)

    # --- Stage 4: Compile source citations ---
    # Deduplicate by (source, page) pairs
    seen = set()
    sources = []
    for chunk in retrieved:
        key = (chunk['source'], chunk['page'])
        if key not in seen:
            seen.add(key)
            sources.append({
                'file': chunk['source'],
                'page': chunk['page'],
                'doc_type': chunk['doc_type'],
                'confidence': chunk['confidence']
            })

    # --- Stage 5: Compute aggregate confidence ---
    avg_conf = round(np.mean([c['confidence'] for c in retrieved]), 3)

    return {
        'answer':         answer,
        'sources':        sources,
        'chunks':         retrieved,
        'chunk_count':    len(retrieved),
        'avg_confidence': avg_conf
    }


print("✅ run_rag_pipeline() defined.")

✅ run_rag_pipeline() defined.


## Step 10: Document Upload Handler

This function handles new file uploads from the Gradio UI:
1. Extracts text (digital or OCR)
2. Chunks with metadata
3. Appends to global chunk store
4. Rebuilds the FAISS index with all documents

Rebuilding the index on each upload keeps the implementation simple. For production with 1000+ docs, incremental index updates would be preferred.

In [11]:
def process_uploaded_pdf(file_path: str) -> str:
    """
    Process an uploaded PDF file:
    1. Extract text (with OCR fallback)
    2. Chunk and tag with metadata
    3. Rebuild FAISS index

    Returns a status message for display in the UI.
    """
    global all_chunks, faiss_index

    if file_path is None:
        return "⚠️ No file uploaded."

    source_name = Path(file_path).name

    # Avoid duplicate uploads
    existing_sources = {c['source'] for c in all_chunks}
    if source_name in existing_sources:
        return f"⚠️ '{source_name}' is already indexed."

    try:
        # Step 1 — Extract text
        pages = extract_text_from_pdf(file_path)

        # Step 2 — Chunk with metadata
        new_chunks = chunk_pages(pages, source_name)

        if not new_chunks:
            return f"❌ Could not extract any text from '{source_name}'."

        # Step 3 — Add to global store and rebuild FAISS index
        all_chunks.extend(new_chunks)
        faiss_index = build_faiss_index(all_chunks)

        total_pages = len(pages)
        ocr_pages = sum(1 for p in pages if p['method'] == 'ocr')

        return (
            f"✅ Indexed '{source_name}'\n"
            f"   Pages: {total_pages} ({ocr_pages} OCR) | "
            f"Chunks: {len(new_chunks)} | "
            f"Total indexed: {len(all_chunks)} chunks across {len(existing_sources)+1} doc(s)"
        )

    except Exception as e:
        return f"❌ Error processing '{source_name}': {str(e)}"


def format_response_for_ui(result: Dict) -> Tuple[str, str]:
    """
    Format the RAG pipeline result into two strings for the Gradio UI:
    - answer_text: The main answer with inline confidence note
    - sources_text: Formatted source citations
    """
    answer = result['answer']
    conf_pct = int(result['avg_confidence'] * 100)
    chunk_count = result['chunk_count']

    answer_text = (
        f"{answer}\n\n"
        f"---\n"
        f"📊 Retrieval stats: {chunk_count} chunk(s) used | "
        f"Avg. confidence: {conf_pct}%"
    )

    if result['sources']:
        lines = ["**Sources cited:**"]
        for s in result['sources']:
            lines.append(
                f"• 📄 {s['file']} — Page {s['page']} "
                f"[{s['doc_type']}] (conf: {int(s['confidence']*100)}%)"
            )
        sources_text = "\n".join(lines)
    else:
        sources_text = "No sources found."

    return answer_text, sources_text


print("✅ process_uploaded_pdf() and format_response_for_ui() defined.")

✅ process_uploaded_pdf() and format_response_for_ui() defined.


## Step 11: Gradio User Interface

We build a clean, product-demo-style Gradio UI with:
- **Upload panel** — drag & drop PDF upload with index status
- **Chat interface** — question input, conversation history
- **Source panel** — shows which documents and pages were cited
- **Filter** — optional document type filter for targeted retrieval

Run this cell to launch the UI in Colab (a public share link will appear).

In [12]:
# --- Gradio chat handler ---
chat_history = []  # Stores (user_message, bot_response) tuples

def chat(user_message: str, history: list, doc_filter: str) -> Tuple[list, str, str]:
    """Handle a chat turn: retrieve → generate → format → return updated UI state."""
    if not user_message.strip():
        return history, "", ""

    # Map UI filter label to doc_type value (or None for 'All')
    filter_map = {
        "All Documents": None,
        "Clinical": "clinical",
        "Regulatory": "regulatory",
        "Drug Label": "label",
        "Research": "research",
    }
    active_filter = filter_map.get(doc_filter, None)

    # Run the full RAG pipeline
    result = run_rag_pipeline(user_message, doc_type_filter=active_filter)
    answer_text, sources_text = format_response_for_ui(result)

    # Append to chat history
    history.append((user_message, answer_text))
    return history, sources_text, ""


def clear_chat():
    """Reset the chat history."""
    return [], "", ""


# --- Build the Gradio app ---
with gr.Blocks(
    title="Pfizer Document Intelligence",
    theme=gr.themes.Soft(primary_hue="blue"),
    css=".source-box { font-size: 0.88rem; background: #f0f4ff; border-radius: 8px; padding: 10px; }"
) as demo:

    gr.Markdown(
        """# 🧬 Pfizer Document Intelligence
        **RAG-powered Q&A over pharmaceutical documents**
        Upload PDFs (digital or scanned) → Ask questions → Get cited answers
        """
    )

    with gr.Row():
        # ── Left column: Upload + status ──────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 📁 Document Upload")
            file_input = gr.File(
                label="Upload PDF (digital or scanned)",
                file_types=[".pdf"],
                type="filepath"
            )
            upload_btn = gr.Button("📤 Index Document", variant="primary")
            upload_status = gr.Textbox(
                label="Index Status",
                lines=3,
                interactive=False,
                placeholder="Upload a PDF and click 'Index Document' to begin."
            )

            gr.Markdown("### 🔍 Filter by Document Type")
            doc_filter = gr.Radio(
                choices=["All Documents", "Clinical", "Regulatory", "Drug Label", "Research"],
                value="All Documents",
                label=""
            )

            gr.Markdown("### 📌 Cited Sources")
            sources_box = gr.Markdown(
                value="Sources will appear here after asking a question.",
                elem_classes=["source-box"]
            )

        # ── Right column: Chat interface ───────────────────────────────────────
        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask a Question")
            chatbot = gr.Chatbot(
                label="Conversation",
                height=420,
                bubble_full_width=False
            )
            with gr.Row():
                user_input = gr.Textbox(
                    placeholder="Ask about your documents… e.g. 'What are the contraindications?'",
                    label="",
                    scale=5
                )
                send_btn = gr.Button("Send ➤", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ Clear Chat", size="sm")

            gr.Markdown(
                "*Answers are grounded in uploaded documents only. "
                "Confidence scores reflect semantic similarity.*"
            )

    # ── Wire up event handlers ─────────────────────────────────────────────────
    upload_btn.click(
        fn=process_uploaded_pdf,
        inputs=[file_input],
        outputs=[upload_status]
    )

    send_btn.click(
        fn=chat,
        inputs=[user_input, chatbot, doc_filter],
        outputs=[chatbot, sources_box, user_input]
    )

    user_input.submit(  # Also trigger on Enter key
        fn=chat,
        inputs=[user_input, chatbot, doc_filter],
        outputs=[chatbot, sources_box, user_input]
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, sources_box, user_input]
    )

# Launch the app (share=True gives a public Colab URL)
print("🚀 Launching Gradio app...")
demo.launch(share=True, debug=False)

/tmp/ipykernel_1776/1248391764.py:34: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1776/1248391764.py:34: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1776/1248391764.py:80: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1776/1248391764.py:80: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1776/124

🚀 Launching Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d40baafddad24a84c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 12: Quick Test (without UI)

Run this cell to test the pipeline programmatically with a sample PDF.
Useful for debugging before using the UI.

In [13]:
# ── Quick headless test ────────────────────────────────────────────────────────
# To test without uploading via the UI:
# 1. Upload a PDF to Colab's file system via the Files panel on the left
# 2. Update TEST_PDF_PATH below
# 3. Run this cell

TEST_PDF_PATH = "/content/drive/MyDrive/Colab Notebooks/Project 7/Files/pharma-blob-sample.pdf"  # ← change to your file
TEST_QUERY = "What are the main findings or key points in this document?"

if os.path.exists(TEST_PDF_PATH):
    print("=" * 60)
    print("HEADLESS PIPELINE TEST")
    print("=" * 60)

    # Process the document
    status = process_uploaded_pdf(TEST_PDF_PATH)
    print("\nIndexing status:", status)

    # Run a query
    print(f"\nQuery: {TEST_QUERY}")
    result = run_rag_pipeline(TEST_QUERY)

    print(f"\n[ANSWER]")
    print(result['answer'])

    print(f"\n[RETRIEVAL STATS]")
    print(f"  Chunks used     : {result['chunk_count']}")
    print(f"  Avg confidence  : {int(result['avg_confidence']*100)}%")

    print(f"\n[SOURCES]")
    for s in result['sources']:
        print(f"  • {s['file']} | Page {s['page']} | Type: {s['doc_type']} | Conf: {int(s['confidence']*100)}%")
else:
    print(f"⚠️  Test file not found at '{TEST_PDF_PATH}'.")
    print("   Upload a PDF via the Colab Files panel and update TEST_PDF_PATH above.")

HEADLESS PIPELINE TEST
📄 Processing 'pharma-blob-sample.pdf' — 10 page(s)
   ✅ Extracted: 10 digital page(s), 0 OCR page(s)
   📦 Created 35 chunks from 'pharma-blob-sample.pdf' (type: regulatory)
⏳ Embedding 35 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built — 35 vectors stored.

Indexing status: ✅ Indexed 'pharma-blob-sample.pdf'
   Pages: 10 (0 OCR) | Chunks: 35 | Total indexed: 35 chunks across 1 doc(s)

Query: What are the main findings or key points in this document?

[ANSWER]
BSE-RA-2023-AKTA-001 This declaration is based on information provided by our raw material suppliers and our own assessment of the manufacturing process

[RETRIEVAL STATS]
  Chunks used     : 4
  Avg confidence  : 92%

[SOURCES]
  • pharma-blob-sample.pdf | Page 1 | Type: regulatory | Conf: 92%
  • pharma-blob-sample.pdf | Page 6 | Type: regulatory | Conf: 92%
  • pharma-blob-sample.pdf | Page 10 | Type: regulatory | Conf: 92%


Comprehensive Testing

In [17]:
# ── Evaluation Block ───────────────────────────────────────────────────────────
import time

# ── Test Set ──────────────────────────────────────────────────────────────────
# 10 Pharmaceutical product questions + keywords
# retrieval_keywords : words that should appear in a relevant retrieved chunk
# answer_keywords    : words that should appear in a correct generated answer

TEST_SET = [
    {
        "question": "What is the expiration date of this product?",
        "retrieval_keywords": ["expir", "expiry", "use by", "best before"],
        "answer_keywords":    ["expir", "date", "month", "year"],
    },
    {
        "question": "What is the name of this product?",
        "retrieval_keywords": ["product name", "brand", "trade name", "name"],
        "answer_keywords":    ["name", "brand", "product"],
    },
    {
        "question": "What are the storage conditions for this product?",
        "retrieval_keywords": ["storage", "store", "temperature", "refrigerate", "cool", "dry"],
        "answer_keywords":    ["store", "temperature", "cool", "refrigerate"],
    },
    {
        "question": "What is the batch or lot number?",
        "retrieval_keywords": ["batch", "lot", "lot number", "batch number"],
        "answer_keywords":    ["batch", "lot", "number"],
    },
    {
        "question": "Who is the manufacturer of this product?",
        "retrieval_keywords": ["manufacturer", "manufactured by", "company", "produced by"],
        "answer_keywords":    ["manufacturer", "company", "manufactured"],
    },
    {
        "question": "What is the product's shelf life?",
        "retrieval_keywords": ["shelf life", "stability", "valid", "months", "years"],
        "answer_keywords":    ["shelf life", "months", "years", "stable"],
    },
    {
        "question": "What are the ingredients or composition of this product?",
        "retrieval_keywords": ["ingredient", "composition", "contains", "active", "excipient"],
        "answer_keywords":    ["ingredient", "contains", "composition"],
    },
    {
        "question": "What is the product's packaging type or format?",
        "retrieval_keywords": ["package", "packaging", "bottle", "blister", "vial", "container"],
        "answer_keywords":    ["package", "bottle", "blister", "vial"],
    },
    {
        "question": "What is the country of origin or manufacturing site?",
        "retrieval_keywords": ["country", "origin", "facility", "site", "plant", "manufactured in"],
        "answer_keywords":    ["country", "facility", "site", "origin"],
    },
    {
        "question": "What regulatory or certification marks does this product carry?",
        "retrieval_keywords": ["certified", "certification", "approved", "compliant", "gmp", "iso"],
        "answer_keywords":    ["certified", "approved", "compliant", "certification"],
    },
]

# ── Helpers ───────────────────────────────────────────────────────────────────
def chunk_is_relevant(chunk_text: str, keywords: List[str]) -> bool:
    text_lower = chunk_text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

def answer_is_correct(answer: str, keywords: List[str]) -> bool:
    ans_lower = answer.lower()
    return any(kw.lower() in ans_lower for kw in keywords)

# ── PDF Processing Time ───────────────────────────────────────────────────────
# Set this to the path of your uploaded document
PDF_PATH = "/content/drive/MyDrive/Colab Notebooks/Project 7/Files/pharma-blob-sample.pdf"

print("⏳ Measuring PDF processing time...")
t_proc_start = time.time()
test_pages = extract_text_from_pdf(PDF_PATH)
t_proc_end = time.time()

total_proc_time = t_proc_end - t_proc_start
num_pages = len(test_pages)
time_per_page = total_proc_time / num_pages if num_pages > 0 else 0

print(f"   Pages processed : {num_pages}")
print(f"   Total time      : {total_proc_time:.2f}s")
print(f"   Per-page time   : {time_per_page:.2f}s per page")
print()

# ── Run Evaluation ────────────────────────────────────────────────────────────
print("=" * 60)
print("  RUNNING EVALUATION")
print(f"  Test set : {len(TEST_SET)} questions  |  TOP_K = {TOP_K}")
print("=" * 60)

retrieval_hits   = []
reciprocal_ranks = []
precision_at_k   = []
recall_at_k      = []
answer_correct   = []
retrieval_times  = []
generation_times = []

for i, item in enumerate(TEST_SET):
    q       = item["question"]
    ret_kws = item["retrieval_keywords"]
    ans_kws = item["answer_keywords"]

    # ── Retrieval ─────────────────────────────────────────────────────────────
    t0 = time.time()
    retrieved = retrieve_chunks(q, k=TOP_K)
    retrieval_times.append(time.time() - t0)

    relevance = [chunk_is_relevant(c["text"], ret_kws) for c in retrieved]
    hit       = int(any(relevance))
    retrieval_hits.append(hit)
    recall_at_k.append(hit)
    precision_at_k.append(sum(relevance) / len(relevance) if relevance else 0)

    rr = 0.0
    for rank, rel in enumerate(relevance, start=1):
        if rel:
            rr = 1.0 / rank
            break
    reciprocal_ranks.append(rr)

    # ── Generation ────────────────────────────────────────────────────────────
    try:
        prompt = build_prompt(q, retrieved)
        t1 = time.time()
        answer_text = llm_generate(prompt, max_new_tokens=150)
        generation_times.append(time.time() - t1)
        correct = int(answer_is_correct(answer_text, ans_kws))
    except Exception as e:
        answer_text = ""
        correct = 0
        generation_times.append(0)
        print(f"  ⚠️  Generation error on Q{i+1}: {e}")

    answer_correct.append(correct)

    r_icon = "✅" if hit    else "❌"
    a_icon = "✅" if correct else "❌"
    print(f"  Q{i+1:02d}  Retrieval {r_icon}  MRR={rr:.2f}  Answer {a_icon}")

# ── Results Summary ────────────────────────────────────────────────────────────
n = len(TEST_SET)
print("\n" + "=" * 60)
print("  RETRIEVAL PERFORMANCE")
print("=" * 60)
print(f"  Hit Rate     : {np.mean(retrieval_hits)*100:.1f}%")
print(f"  Recall@{TOP_K}     : {np.mean(recall_at_k)*100:.1f}%")
print(f"  Precision@{TOP_K}  : {np.mean(precision_at_k)*100:.1f}%")
print(f"  MRR          : {np.mean(reciprocal_ranks):.3f}")
print(f"  Avg latency  : {np.mean(retrieval_times)*1000:.1f} ms per query")

print(f"\n  PDF PROCESSING")
print(f"  Per-page time  : {time_per_page:.2f}s per page  ({num_pages} pages total)")

print("\n" + "=" * 60)
print("  END-TO-END ACCURACY")
print("=" * 60)
print(f"  Answer accuracy : {np.mean(answer_correct)*100:.1f}%  ({sum(answer_correct)}/{n} correct)")
print(f"  Avg gen time    : {np.mean(generation_times):.2f}s per query")
print(f"  Avg total time  : {(np.mean(retrieval_times) + np.mean(generation_times)):.2f}s per query")
print("=" * 60)
print(f"\n  ⚠️  Evaluated on {n} manually annotated questions from 1 document.")
print("  Results are illustrative, not generalisable across documents.")

⏳ Measuring PDF processing time...
📄 Processing 'pharma-blob-sample.pdf' — 10 page(s)
   ✅ Extracted: 10 digital page(s), 0 OCR page(s)
   Pages processed : 10
   Total time      : 1.19s
   Per-page time   : 0.12s per page

  RUNNING EVALUATION
  Test set : 10 questions  |  TOP_K = 4
  Q01  Retrieval ✅  MRR=0.33  Answer ❌
  Q02  Retrieval ❌  MRR=0.00  Answer ❌
  Q03  Retrieval ✅  MRR=0.50  Answer ❌
  Q04  Retrieval ✅  MRR=1.00  Answer ✅
  Q05  Retrieval ❌  MRR=0.00  Answer ❌
  Q06  Retrieval ✅  MRR=1.00  Answer ✅
  Q07  Retrieval ❌  MRR=0.00  Answer ❌
  Q08  Retrieval ✅  MRR=1.00  Answer ❌
  Q09  Retrieval ✅  MRR=1.00  Answer ❌
  Q10  Retrieval ✅  MRR=0.50  Answer ❌

  RETRIEVAL PERFORMANCE
  Hit Rate     : 70.0%
  Recall@4     : 70.0%
  Precision@4  : 35.0%
  MRR          : 0.533
  Avg latency  : 12.5 ms per query

  PDF PROCESSING
  Per-page time  : 0.12s per page  (10 pages total)

  END-TO-END ACCURACY
  Answer accuracy : 20.0%  (2/10 correct)
  Avg gen time    : 0.93s per query
  

In [37]:
# ── Slide Example Query ───────────────────────────────────────────────────────
example_query = "What regulatory or certification marks does this product carry?"

# Retrieve and deduplicate chunks
raw_chunks = retrieve_chunks(example_query, k=TOP_K)
seen = set()
example_chunks = []
for chunk in raw_chunks:
    key = (chunk['source'], chunk['page'], chunk['text'][:150])
    if key not in seen:
        seen.add(key)
        example_chunks.append(chunk)

# Build and run
example_prompt = build_prompt(example_query, example_chunks)
example_answer = llm_generate(example_prompt, max_new_tokens=150)

# Print formatted output for your slide
print(f'Query: "{example_query}"\n')
print("Retrieved Chunks:")
for i, chunk in enumerate(example_chunks, 1):
    preview = chunk['text'][:120].replace('\n', ' ').strip()
    similarity = chunk['confidence']
    print(f'  {i}. "{preview}..." (Similarity: {similarity:.2f})')

print(f'\nFinal Answer: "{example_answer}"')

Raw FAISS indices: [18 25 16 28]
Query: "What regulatory or certification marks does this product carry?"

Retrieved Chunks:
  1. "SE/TSE risk assessment) - EU Regulation (EC) No 1774/2002 (animal by-products) - United States Pharmacopeia (USP) <87> a..." (Similarity: 0.95)
  2. "ical findings) Next Scheduled Audit: November 2025 2. Certifications Certification Number Valid Until ISO 9001:2015 QMS-..." (Similarity: 0.95)
  3. "Declaration Regarding Transmissible Spongiform Encephalopathies (BSE/TSE Compliance Statement) Manufacturer: Cytiva Swed..." (Similarity: 0.94)
  4. "Information: Sub-Supplier Material Provided Qualification Status DuPont (Tyvek) Lid film (1073B) Approved (2020) Wacker..." (Similarity: 0.94)

Final Answer: "DuPont (Tyvek)"
